# ProcessDataFrame: Frictionless Process Behavior Analysis

This notebook demonstrates the `ProcessDataFrame` API that makes process behavior analysis intuitive and discoverable.

## Key Features

1. **Auto-completion** for column names via `data.columns.ColumnName`
2. **Two-step workflow**: `formulate()` creates a Study, then `analyze()` runs charts
3. **SDS-driven execution** - the data dictates what analyses are supported
4. **Automatic chart recommendations** based on detected structure
5. **Clear explanations** of what's running and why

In [1]:
import sys
print(sys.executable)
print(sys.path)

/Users/nicholas/Documents/projects/processbehavior/venv/bin/python3.13
['/Users/nicholas/Documents/projects/processbehavior/examples/${PYTHONPATH}', '/Users/nicholas/Documents/projects/processbehavior/examples/${workspaceFolder}', '/Users/nicholas/Documents/projects/processbehavior/examples', '/usr/local/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python313.zip', '/usr/local/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python3.13', '/usr/local/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python3.13/lib-dynload', '', '/Users/nicholas/Documents/projects/processbehavior/venv/lib/python3.13/site-packages', '/Users/nicholas/Documents/projects/processbehavior']


In [2]:
import numpy as np
import pandas as pd
from processbehavior import ProcessBehavior


# Set random seed for reproducibility
np.random.seed(42)

## Example 1: Simple Series (SDS 0) → IMR Chart

When you have a simple time series with no grouping structure, ProcessDataFrame detects **SDS 0** and recommends an **Individual Moving Range (IMR)** chart.

The two-step workflow:
1. `formulate()` - analyzes structure and returns a Study with recommendations
2. `analyze()` - runs the selected chart type

In [3]:
# Create simple measurement series
simple_data = pd.DataFrame({
    'Measurement': np.random.normal(100, 2, 30),
    'Time': pd.date_range('2024-01-01', periods=30, freq='D')
})

# Wrap in ProcessBehavior
pb = ProcessBehavior(simple_data)

# Step 1: formulate() - analyze structure and get recommendations
# Auto-completion magic! Type `pb.cols.` and see your columns appear
study = pb.formulate(
    response=pb.cols.Measurement,
    time=pb.cols.Time
)

print(study)
print(f"\nRecommended chart: {study.recommended_chart}")
print(f"Valid charts: {study.valid_charts}")

╔══════════════════════════════════════════════════════════════════╗
║                        STUDY FORMULATION                         ║
╠══════════════════════════════════════════════════════════════════╣
║  Response: Measurement                                           ║
║  Time:     Time                                                  ║
║  Precision: 3 decimal places                                     ║
╠══════════════════════════════════════════════════════════════════╣
║  Detected: SDS 0 - Simple Series                                 ║
║                                                                  ║
║  Valid Charts:  Imr, R                                           ║
║  Recommended:   Imr                                              ║
╠══════════════════════════════════════════════════════════════════╣
║  Next: study.execute() or study.execute(chart='Xbar')            ║
╚══════════════════════════════════════════════════════════════════╝

Recommended chart: Imr
Valid char

## Example 2: Manufacturing Data with Operators (Grouped) → Xbar/S Charts

When you have rational subgroups with **replication** (multiple measurements per cell), ProcessDataFrame detects the structure and recommends **Xbar and S charts** to track both location (mean) and variation.

**Key**: We create a structured design where each Operator/Machine/Time combination has multiple measurements (n=2). This creates proper subgroups for Xbar/S analysis.

In [4]:
# Create manufacturing data with operators and machines
# Structure: Each operator works on each machine at different times
# This creates rational subgroups (multiple measurements per cell)
n_time_points = 20
n_reps = 2  # 2 measurements per operator/machine/time combination

manufacturing_data = pd.DataFrame({
    'Height': np.random.normal(50, 3, n_time_points * 3 * 3 * n_reps),
    'Operator': np.repeat(['Alice', 'Bob', 'Charlie'], n_time_points * 3 * n_reps),
    'Machine': np.tile(np.repeat(['M1', 'M2', 'M3'], n_time_points * n_reps), 3),
    'ProductionTime': np.tile(np.repeat(pd.date_range('2024-01-01', periods=n_time_points, freq='h'), n_reps), 9)
})

pb = ProcessBehavior(manufacturing_data)

# formulate() with factors for grouped analysis
study = pb.formulate(
    response=pb.cols.Height,
    time=pb.cols.ProductionTime,
    factors=[pb.cols.Operator, pb.cols.Machine]
)

print(study)
print(f"\nDetected SDS: {study.sds}")
print(f"Recommended chart: {study.recommended_chart}")
print(f"Valid charts: {study.valid_charts}")

╔══════════════════════════════════════════════════════════════════╗
║                        STUDY FORMULATION                         ║
╠══════════════════════════════════════════════════════════════════╣
║  Response: Height                                                ║
║  Factors:  Operator, Machine                                     ║
║  Time:     ProductionTime                                        ║
║  Precision: 3 decimal places                                     ║
╠══════════════════════════════════════════════════════════════════╣
║  Detected: SDS 1 - Full Factorial with Complete Replication      ║
║                                                                  ║
║  Valid Charts:  Xbar, S, R, Imr                                  ║
║  Recommended:   Xbar                                             ║
║  Residuals:     R2_S, R3_Xbar, R3_S, R4_Xbar, R4_S, R5_Xbar, R5_S║
╠══════════════════════════════════════════════════════════════════╣
║  Next: study.execute() or study.

In [5]:
# Example with irregular/random data (will detect SDS 6)
n_obs = 120
irregular_data = pd.DataFrame({
    'Height': np.random.normal(50, 3, n_obs),
    'Width': np.random.normal(25, 1, n_obs),
    'Operator': np.random.choice(['Alice', 'Bob', 'Charlie'], n_obs),
    'Machine': np.random.choice(['M1', 'M2', 'M3'], n_obs),
    'ProductionTime': pd.date_range('2024-01-01', periods=n_obs, freq='h')
})

pb = ProcessBehavior(irregular_data)

# formulate() detects irregular structure
study = pb.formulate(
    response=pb.cols.Height,
    time=pb.cols.ProductionTime,
    factors=[pb.cols.Operator, pb.cols.Machine]
)

print(f"Detected SDS: {study.sds}")
print(f"Recommended chart: {study.recommended_chart}")

Detected SDS: 1
Recommended chart: Xbar


In [6]:
# Step 2: execute() runs the chart and returns results
result = study.execute()

print("Analysis Result:")
print(result)
print(f"\nCharts available: {result.all_charts}")

Analysis Result:
ANALYSIS RESULT SUMMARY

Sampling Design State: SDS 1
Description: Full replication (all cells n≥2)

Analysis Type: Xbar
Response Variable: Height
Grouping: Operator, Machine
Time Variable: ProductionTime

Observations: 120
Charts: Xbar, Sbar

Capabilities:
  Residuals: ✓
  Effects: ✓
  Interactions: ✓

Charts available: ['Xbar', 'S']


## Example 3: Quality Data with Multiple Factors

ProcessDataFrame handles complex multi-factor designs and explains the detected Sampling Design State.

In [7]:
# Create data with 2 factors and time
quality_data = pd.DataFrame({
    'Strength': np.random.normal(100, 5, 120),
    'Temperature': np.random.choice(['Low', 'High'], 120),
    'Pressure': np.random.choice(['A', 'B'], 120),
    'Batch': range(1, 121)
})

pb = ProcessBehavior(quality_data)

study = pb.formulate(
    response=pb.cols.Strength,
    time=pb.cols.Batch,
    factors=[pb.cols.Temperature, pb.cols.Pressure]
)

print(f"SDS: {study.sds}")
print(f"Recommended: {study.recommended_chart}")
print(f"Valid charts: {study.valid_charts}")

SDS: 1
Recommended: Xbar
Valid charts: ['Xbar', 'S', 'R', 'Imr']


## Example 4: Using the Charts Accessor

The Study object provides a `charts` accessor for IDE auto-completion of valid chart types.

In [8]:
# Create simple data
simple_data = pd.DataFrame({
    'Value': np.random.normal(100, 10, 50),
    'Sequence': range(1, 51)
})

pb = ProcessBehavior(simple_data)
study = pb.formulate(
    response=pb.cols.Value,
    time=pb.cols.Sequence
)

# Use the charts accessor for IDE auto-completion
print(f"Valid charts: {study.valid_charts}")
print(f"Access via study.charts.Imr: {study.charts.Imr}")

# Execute using the charts accessor
result = study.execute(chart=study.charts.Imr)
print(f"\nAnalysis complete: {result.all_charts}")

Valid charts: ['Imr', 'R']
Access via study.charts.Imr: Imr

Analysis complete: ['Imr']


## Why This API is Better

### Before (Old API)
```python
# Easy to make typos in column names!
spec = {
    'analysis_type': 'Imr',  # User must know which type
    'response_var': 'Measurment',  # Typo! Will fail
    'time_var': 'Time',
    'rsg_vars': None
}
analysis = Analysis(df, spec)
```

### After (New API)
```python
# Auto-completion prevents typos!
# System chooses correct analysis type!
data = ProcessDataFrame(df)

# Step 1: formulate() - analyze structure
study = data.formulate(
    response=data.columns.Measurement,  # IDE autocompletes!
    time=data.columns.Time
)

# Step 2: analyze() - run charts
result = study.analyze()
```

## Benefits

1. **No more typos** - IDE autocomplete ensures valid column names
2. **No more wrong analysis types** - System detects SDS and recommends appropriately
3. **Two-step workflow** - Inspect recommendations before running analysis
4. **Transparency** - Study object shows SDS, valid charts, and recommendations
5. **Follows the data** - Analysis adapts to your data structure
6. **Pythonic** - Clean, readable, discoverable API

## Understanding Sampling Design States (SDS)

ProcessDataFrame automatically detects which of 7 SDS categories your data falls into:

- **SDS 0**: No grouping or time → IMR chart
- **SDS 1**: Full replication (all cells n≥2) → Full VAS + Xbar/S
- **SDS 2**: No replication (all cells n=1) → Residuals + Xbar/S
- **SDS 3**: Partial replication (mixed) → Hybrid approach
- **SDS 4**: Single condition over time → Time series analysis
- **SDS 5**: Nested design → Variance components
- **SDS 6**: Irregular grid → Adaptive limits

You don't need to know these - the system figures it out and tells you!